<h1 style="
    color:#FFFFFF;
    background:linear-gradient(90deg,#0F766E,#0284C7);
    text-align:center;
    font-weight:bold;
    padding:18px 12px;
    border-radius:10px;
    margin-bottom:7px;">
    Week 9 — Day 1: Sprint 4 Planning, Serialization & MLOps
</h1>

<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">
    From a Trained IMDb Model to a Deployment-Ready Project
</h3>

<div style="border-left:6px solid #FB7185;background-color:#FFF1F2;padding:12px 16px;margin:16px 0;border-radius:6px;color:#7F1D1D;">
<b>Week 9 — Model Deployment • Sprint 4 • Day 1</b><br>
Project used today: <b>IMDb Sentiment Analysis</b> from Week 8.<br>
Week 8 result: TF-IDF + Logistic Regression, with the saved vectorizer and trained model prepared for deployment.
</div>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p><b>Main question:</b></p>
<div style="font-size:1.12em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:16px;border-radius:8px;margin:14px 0;color:#111827;">
How do we take the model we finished in Week 8 and prepare it to be used by a real application without retraining it every time?
</div>
</div>

<a id="toc"></a>
<h2 style="color:#92400E; background-color:#FBBF24; font-weight:bold; margin-top:30px; padding:9px 14px; border-radius:7px;">Table of Contents</h2>
<div style="border:1px solid #CBD5E1; padding:16px 25px; border-radius:8px; background-color:#FFFFFF; color:#111827;">
<ul style="font-weight:bold; line-height:1.95; color:#1F2937;">
<li><a href="#section0">0. Setup — Imports, Paths & Reproducibility</a></li>
<li><a href="#section1">1. Day 1 Learning Map</a></li>
<li><a href="#section2">2. Sprint 4 Planning</a></li>
<li><a href="#section3">3. Why Deployment Matters</a></li>
<li><a href="#section4">4. Model Serialization</a></li>
<li><a href="#section5">5. Bring the Week 8 Artifacts into Day 1</a></li>
<li><a href="#section6">6. Load the Saved Vectorizer & Model</a></li>
<li><a href="#section7">7. Keep Training and Serving Preprocessing Identical</a></li>
<li><a href="#section8">8. Verify a Known Prediction</a></li>
<li><a href="#section9">9. Build a Small Deployment Prediction Function</a></li>
<li><a href="#section10">10. Reproducibility — requirements.txt & Fixed Seeds</a></li>
<li><a href="#section11">11. MLflow & Experiment Traceability</a></li>
<li><a href="#section12">12. Day 1 Definition of Done</a></li>
<li><a href="#section13">13. What I Learned Today</a></li>
<li><a href="#section14">14. Day 2 Preview — FastAPI</a></li>
</ul>
</div>

<a id="section0"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">0. Setup — Imports, Paths & Reproducibility</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">0.1 Expected folder structure</h3>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">The Main Folder/
├── Week 8/
│   └── Day 2/
│       ├── IMDB_Dataset_Cleaned_Day1.csv
│       ├── tfidf_vectorizer.joblib
│       └── sentiment_model.joblib
└── Week 9/
    └── Day 1/
        ├── Day1.ipynb
        └── preprocessing.py</div>
<p>The notebook searches both the current folder and the Week 8 Day 2 folder, so it remains easy to run from VS Code.</p>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Important:</b> Run this notebook from <code>Week 9/Day 1</code>. Keep <code>preprocessing.py</code> beside it.</div>
</div>

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import random
import shutil
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def locate_day1_dir():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "Week 9" / "Day 1",
        cwd.parent / "Week 9" / "Day 1",
        cwd.parent.parent / "Week 9" / "Day 1",
    ]

    for folder in candidates:
        if (folder / "preprocessing.py").exists():
            return folder.resolve()

    # VS Code may start the kernel from the workspace root.
    matches = list(cwd.glob("**/Week 9/Day 1/preprocessing.py"))
    if matches:
        return matches[0].parent.resolve()

    # Fallback: use the current working directory and show a clear warning.
    return cwd

DAY1_DIR = locate_day1_dir()
if str(DAY1_DIR) not in sys.path:
    sys.path.insert(0, str(DAY1_DIR))


print("Fixed random seed:", SEED)

Kernel working folder: C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week 9\Day 1
Detected Day 1 folder: C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week 9\Day 1
Fixed random seed: 42


<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:10px 15px;margin:10px 0;border-radius:4px;color:#134E4A;">
<b>Reproducibility note:</b> A fixed seed makes random operations repeatable. It does not make every possible machine-learning operation perfectly deterministic, but it is an important reproducibility practice.
</div>

In [2]:
# Resources required by preprocessing.py
import nltk

resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print("NLTK resources checked.")

NLTK resources checked.


<a id="section1"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">1. Day 1 Learning Map</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>Week 8 finished with a trained and evaluated sentiment model. Day 1 of Week 9 does <b>not</b> build a new model. It prepares the existing model for deployment.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Week 8
Clean Reviews
   ↓
TF-IDF Vectorizer
   ↓
Logistic Regression
   ↓
Evaluated Trained Model

Week 9 — Day 1
   ↓
Serialize / organize artifacts
   ↓
Load them without retraining
   ↓
Verify same preprocessing
   ↓
Reproduce a known prediction
   ↓
Prepare requirements & experiment traceability
   ↓
Ready for FastAPI</div>

<table style="width:100%;border-collapse:collapse;margin-top:14px;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Today</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Why</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Sprint planning</td><td style="padding:9px;border:1px solid #CBD5E1;">Define what must be deployed and polished.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Serialization</td><td style="padding:9px;border:1px solid #CBD5E1;">Save and reload the trained components without retraining.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Same preprocessing</td><td style="padding:9px;border:1px solid #CBD5E1;">Prevent training/serving skew.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Reproducibility</td><td style="padding:9px;border:1px solid #CBD5E1;">Make the environment repeatable.</td></tr>
</table>
</div>

<a id="section2"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">2. Sprint 4 Planning</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">2.1 Sprint Goal</h3>
<div style="font-size:1.08em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:14px;border-radius:8px;margin:12px 0;color:#111827;">Deploy the Week 8 sentiment model as a live, public application.</div>
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">2.2 Sprint Backlog</h3>
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Task</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Planned Day</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Output</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Serialize & verify model artifacts</td><td style="padding:9px;border:1px solid #CBD5E1;">Day 1</td><td style="padding:9px;border:1px solid #CBD5E1;">.joblib files + reproducibility files</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Serve with FastAPI</td><td style="padding:9px;border:1px solid #CBD5E1;">Day 2</td><td style="padding:9px;border:1px solid #CBD5E1;">POST /predict endpoint</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Build Streamlit UI</td><td style="padding:9px;border:1px solid #CBD5E1;">Day 3</td><td style="padding:9px;border:1px solid #CBD5E1;">Interactive dashboard</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Public deployment</td><td style="padding:9px;border:1px solid #CBD5E1;">Day 4</td><td style="padding:9px;border:1px solid #CBD5E1;">Live public URL</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;">Repository polish</td><td style="padding:9px;border:1px solid #CBD5E1;">Day 5</td><td style="padding:9px;border:1px solid #CBD5E1;">README, technical write-up, final review</td></tr>
</table>
</div>

<a id="section3"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">3. Why Deployment Matters</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>A trained model inside a notebook is useful to the developer, but a real user should not need Jupyter, Python code, or <code>model.predict()</code>.</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Notebook only
Developer → Python code → model.predict()

Deployment
User → Application → Preprocessing → Model → Prediction</div>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Key idea:</b> Deployment turns a trained model into a usable service or application.</div>
</div>

<a id="section4"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">4. Model Serialization</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">4.1 Definition</h3>
<p><b>Serialization</b> means saving a trained Python object to disk so another program can load it later without retraining it.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">During training:
model.fit(...)
      ↓
trained model in memory
      ↓
joblib.dump(...)
      ↓
sentiment_model.joblib

During deployment:
sentiment_model.joblib
      ↓
joblib.load(...)
      ↓
ready model — no retraining</div>
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">4.2 What must be saved for our IMDb project?</h3>
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Artifact</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Why</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>tfidf_vectorizer.joblib</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Contains the TF-IDF vocabulary and learned representation settings.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>sentiment_model.joblib</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Contains the trained Logistic Regression classifier.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><code>preprocessing.py</code></td><td style="padding:9px;border:1px solid #CBD5E1;">Applies the same text cleaning used before TF-IDF in Week 8.</td></tr>
</table>
</div>

<div style="border-left:5px solid #0284C7;background-color:#E0F2FE;padding:11px 15px;margin:12px 0;border-radius:5px;color:#0C4A6E;">
<b>Reference — the save operation we already performed in the Week 8 notebook:</b>
<pre style="background:#FFFFFF;padding:10px;border-radius:6px;">joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(model, "sentiment_model.joblib")</pre>
Day 1 now verifies and organizes these artifacts for deployment.
</div>

<a id="section5"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">5. Bring the Week 8 Artifacts into Day 1</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>For deployment, it is cleaner to keep the final model artifacts with the Week 9 deployment code. The next cell searches for the two files and copies them into the current Day 1 folder if necessary.</p>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>If a file is not found:</b> return to Week 8 Day 2, run the TF-IDF and Logistic Regression cells, then run the <code>joblib.dump()</code> save cell.</div>
</div>

In [ ]:
def find_file(filename):
    candidates = [
        DAY1_DIR / filename,
        DAY1_DIR.parent.parent / "Week 8" / "Day 2" / filename,
        Path("Week 8") / "Day 2" / filename,
        Path("..") / ".." / "Week 8" / "Day 2" / filename,
    ]
    for path in candidates:
        path = path.resolve()
        if path.exists():
            return path
    return None

model_source = find_file("sentiment_model.joblib")
vectorizer_source = find_file("tfidf_vectorizer.joblib")

if model_source is None or vectorizer_source is None:
    raise FileNotFoundError(
        "Could not find the saved model/vectorizer. "
        "Run the joblib.dump() cell in Week 8/Day 2 first."
    )

MODEL_PATH = DAY1_DIR / "sentiment_model.joblib"
VECTORIZER_PATH = DAY1_DIR / "tfidf_vectorizer.joblib"

if model_source != MODEL_PATH.resolve():
    shutil.copy2(model_source, MODEL_PATH)
if vectorizer_source != VECTORIZER_PATH.resolve():
    shutil.copy2(vectorizer_source, VECTORIZER_PATH)


print("Deployment artifacts are now in Day 1.")

Model artifact     : C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week 9\Day 1\sentiment_model.joblib
Vectorizer artifact: C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week 9\Day 1\tfidf_vectorizer.joblib
Deployment artifacts are now in Day 1.


<a id="section6"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">6. Load the Saved Vectorizer & Model</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>This is the deployment payoff of serialization. We load the trained objects directly from disk. Notice that there is no <code>fit()</code> in this section.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">joblib files
   ↓
joblib.load()
   ↓
loaded_vectorizer + loaded_model
   ↓
ready for prediction</div>
</div>

In [4]:
loaded_vectorizer = joblib.load(VECTORIZER_PATH)
loaded_model = joblib.load(MODEL_PATH)

print("Vectorizer type:", type(loaded_vectorizer))
print("Model type     :", type(loaded_model))
print("Loaded successfully — no retraining was performed.")

Vectorizer type: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Model type     : <class 'sklearn.linear_model._logistic.LogisticRegression'>
Loaded successfully — no retraining was performed.


In [5]:
# Small sanity check of the learned TF-IDF vocabulary
print("Vocabulary size:", len(loaded_vectorizer.get_feature_names_out()))
print("Example features:", loaded_vectorizer.get_feature_names_out()[:15].tolist())

Vocabulary size: 5000
Example features: ['abandon', 'abbott', 'abc', 'ability', 'able', 'abraham', 'abrupt', 'absence', 'absent', 'absolute', 'absolutely', 'absurd', 'abuse', 'abusive', 'abysmal']


<a id="section7"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">7. Keep Training and Serving Preprocessing Identical</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">7.1 Training / Serving Skew</h3>
<p><b>Training/serving skew</b> happens when the model receives data prepared differently during deployment than during training.</p>
<div style="background-color:#FFF7ED;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Correct deployment path
Raw review
   ↓
Same Week 8 preprocessing
   ↓
Saved TF-IDF vectorizer.transform()
   ↓
Saved Logistic Regression model.predict()
   ↓
Sentiment</div>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Rule:</b> Do not call <code>fit()</code> or <code>fit_transform()</code> on user input. Training has already happened.</div>
</div>

In [6]:
PREPROCESSING_PATH = DAY1_DIR / "preprocessing.py"
if not PREPROCESSING_PATH.exists():
    raise FileNotFoundError(
        "preprocessing.py was not found beside Day1.ipynb. "
        "Place the Week 8 text preprocessing functions in this file."
    )

from preprocessing import preprocess_to_string

raw_review = "I didn't like this movie!!! It was terrible."
clean_review = preprocess_to_string(raw_review)

print("Original review:")
print(raw_review)
print("\nClean review:")
print(clean_review)

Original review:
I didn't like this movie!!! It was terrible.

Clean review:
not like movie terrible


<a id="section8"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">8. Verify a Known Prediction</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>A deployment artifact is not trusted just because it loads. We should verify that it reproduces the behavior of the Week 8 model.</p>
<p>The next cells locate the cleaned IMDb dataset, reconstruct the same 80/20 test split using <code>random_state=42</code>, and evaluate the <b>loaded</b> model. We are not training again.</p>
</div>

In [ ]:
def find_clean_csv():
    candidates = [
        DAY1_DIR / "IMDB_Dataset_Cleaned_Day1.csv",
        DAY1_DIR.parent.parent / "Week 8" / "Day 2" / "IMDB_Dataset_Cleaned_Day1.csv",
        Path("Week 8") / "Day 2" / "IMDB_Dataset_Cleaned_Day1.csv",
        Path("..") / ".." / "Week 8" / "Day 2" / "IMDB_Dataset_Cleaned_Day1.csv",
    ]
    for path in candidates:
        path = path.resolve()
        if path.exists():
            return path
    return None

DATA_PATH = find_clean_csv()
if DATA_PATH is None:
    raise FileNotFoundError("Could not find IMDB_Dataset_Cleaned_Day1.csv in Week 8/Day 2.")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns      :", df.columns.tolist())

Dataset path : C:\Users\sadee\OneDrive\Desktop\The Main Folder\The Main Folder\Week 8\Day 2\IMDB_Dataset_Cleaned_Day1.csv
Dataset shape: (5000, 3)
Columns      : ['review', 'clean_review', 'sentiment']


In [8]:
# Reconstruct the same Week 8 split using row indices.
all_indices = np.arange(len(df))
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=df["sentiment"],
)

X_test_loaded = loaded_vectorizer.transform(df.loc[test_idx, "clean_review"])
y_test_loaded = df.loc[test_idx, "sentiment"]
y_pred_loaded = loaded_model.predict(X_test_loaded)

loaded_accuracy = accuracy_score(y_test_loaded, y_pred_loaded)
print("Loaded-model accuracy:", round(loaded_accuracy, 4))
print()
print(classification_report(y_test_loaded, y_pred_loaded))

Loaded-model accuracy: 0.856

              precision    recall  f1-score   support

    negative       0.87      0.83      0.85       496
    positive       0.84      0.88      0.86       504

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



<div style="border-left:5px solid #0F766E;background-color:#CCFBF1;padding:11px 15px;margin:12px 0;border-radius:5px;color:#134E4A;">
<b>Expected check:</b> Week 8 produced an accuracy of approximately <b>0.856</b> on the saved sentiment baseline. A matching result confirms that the serialized model and vectorizer reproduce the Week 8 behavior.
</div>

In [9]:
# Show one known test example: actual label vs loaded-model prediction.
sample_idx = int(test_idx[0])
sample_clean = df.loc[sample_idx, "clean_review"]
sample_actual = df.loc[sample_idx, "sentiment"]
sample_vector = loaded_vectorizer.transform([sample_clean])
sample_predicted = loaded_model.predict(sample_vector)[0]

print("Clean review:")
print(sample_clean[:350])
print("\nActual sentiment   :", sample_actual)
print("Predicted sentiment:", sample_predicted)

Clean review:
movie great venezuelan tourism bird bird bird piranha nice scenery highlight alligator see long boring motorcycle race end caribe drowns definite hollywood prop no definite storyline go venezuelan scenery rip easy rider diamond mining ruthless hunter go crazy reason get end low budget movie could film anywhere outtakes venezuela william smith talen

Actual sentiment   : negative
Predicted sentiment: negative


<a id="section9"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">9. Build a Small Deployment Prediction Function</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>This function represents the exact logic that FastAPI will call on Day 2:</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">raw text
   ↓
preprocess_to_string()
   ↓
loaded_vectorizer.transform()
   ↓
loaded_model.predict() + predict_proba()
   ↓
JSON-ready result</div>
</div>

In [10]:
def predict_sentiment(raw_text):
    clean_text = preprocess_to_string(raw_text)
    vector = loaded_vectorizer.transform([clean_text])

    prediction = loaded_model.predict(vector)[0]
    probabilities = loaded_model.predict_proba(vector)[0]
    classes = loaded_model.classes_

    probability_map = {
        label: float(prob)
        for label, prob in zip(classes, probabilities)
    }

    return {
        "raw_text": raw_text,
        "clean_text": clean_text,
        "prediction": prediction,
        "probabilities": probability_map,
    }

In [11]:
examples = [
    "This movie was amazing. I loved the story and the acting!",
    "I did not enjoy this movie. It was boring and terrible.",
]

for text in examples:
    result = predict_sentiment(text)
    print("Review     :", result["raw_text"])
    print("Clean text :", result["clean_text"])
    print("Prediction :", result["prediction"])
    print("Probabilities:", {k: round(v, 4) for k, v in result["probabilities"].items()})
    print("-" * 90)

Review     : This movie was amazing. I loved the story and the acting!
Clean text : movie amaze love story act
Prediction : positive
Probabilities: {'negative': 0.099, 'positive': 0.901}
------------------------------------------------------------------------------------------
Review     : I did not enjoy this movie. It was boring and terrible.
Clean text : not enjoy movie boring terrible
Prediction : negative
Probabilities: {'negative': 0.868, 'positive': 0.132}
------------------------------------------------------------------------------------------


<a id="section10"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">10. Reproducibility — requirements.txt & Fixed Seeds</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<h3 style="background-color:#F8EEEE;color:#7F1D1D;border:1px solid #D9B9BE;border-left:6px solid #7F1D1D;border-radius:8px;padding:9px 12px;margin:18px 0 12px 0;font-size:1.35em;font-weight:800;line-height:1.35;">10.1 Why pin versions?</h3>
<p>A deployment server creates a fresh Python environment. If package versions are missing or incompatible, code that worked locally may fail on the server.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Local environment
   ↓
requirements.txt with exact versions
   ↓
Deployment environment
   ↓
Same dependencies</div>
<p>The next cell writes a minimal <code>requirements.txt</code> using the versions installed in the environment running this notebook. Days 2 and 3 can add FastAPI, Uvicorn, Pydantic, and Streamlit after they are installed and used.</p>
</div>

In [12]:
required_packages = [
    "joblib",
    "numpy",
    "pandas",
    "scikit-learn",
    "nltk",
]

pinned = []
for package in required_packages:
    try:
        version = metadata.version(package)
        pinned.append(f"{package}=={version}")
    except metadata.PackageNotFoundError:
        print(f"Warning: {package} is not installed in this environment.")

REQUIREMENTS_PATH = DAY1_DIR / "requirements.txt"
REQUIREMENTS_PATH.write_text("\n".join(pinned) + "\n", encoding="utf-8")

print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

joblib==1.5.2
numpy==2.3.5
pandas==2.3.3
scikit-learn==1.7.2
nltk==3.9.2



<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;">
<b>Important:</b> Keep updating <code>requirements.txt</code> as the deployment grows. A requirements file must describe the libraries actually imported by the final application.
</div>

<a id="section11"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">11. MLflow & Experiment Traceability</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p><b>MLflow</b> is used to track experiments, parameters, metrics, and artifacts so we can identify exactly which model version produced a result.</p>
<div style="background-color:#E0F2FE;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">Experiment
├── Model type: Logistic Regression
├── TF-IDF max_features: 5000
├── Random seed: 42
├── Accuracy: ~0.856
└── Artifacts: model + vectorizer + requirements</div>
<p>The cell below is optional if MLflow is not installed in the current environment. It never blocks the rest of the notebook.</p>
</div>

In [13]:
try:
    import mlflow
    MLFLOW_AVAILABLE = True
    print("MLflow version:", mlflow.__version__)
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow is not installed. If your mentor requires logging, install it with: %pip install mlflow")

MLflow is not installed. If your mentor requires logging, install it with: %pip install mlflow


In [14]:
if MLFLOW_AVAILABLE:
    mlflow.set_experiment("week9_sprint4_imdb_deployment")

    with mlflow.start_run(run_name="day1_serialized_sentiment_model"):
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("text_representation", "TF-IDF")
        mlflow.log_param("max_features", len(loaded_vectorizer.get_feature_names_out()))
        mlflow.log_param("random_seed", SEED)
        mlflow.log_metric("loaded_model_accuracy", float(loaded_accuracy))
        mlflow.log_artifact(str(MODEL_PATH))
        mlflow.log_artifact(str(VECTORIZER_PATH))
        mlflow.log_artifact(str(REQUIREMENTS_PATH))

    print("MLflow run logged successfully.")
else:
    print("Skipped MLflow logging; the rest of Day 1 is complete without this optional cell.")

Skipped MLflow logging; the rest of Day 1 is complete without this optional cell.


<a id="section12"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">12. Day 1 Definition of Done</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>Day 1 is complete when the deployment artifacts exist, load correctly, preprocessing is reusable, and a known prediction can be reproduced.</p>
</div>

In [15]:
checks = {
    "Saved model exists": MODEL_PATH.exists(),
    "Saved vectorizer exists": VECTORIZER_PATH.exists(),
    "preprocessing.py exists": PREPROCESSING_PATH.exists(),
    "requirements.txt exists": REQUIREMENTS_PATH.exists(),
    "Model loaded": loaded_model is not None,
    "Vectorizer loaded": loaded_vectorizer is not None,
    "Known evaluation reproduced": loaded_accuracy > 0,
}

check_table = pd.DataFrame(
    [{"Requirement": name, "Status": "PASS" if ok else "FAIL"} for name, ok in checks.items()]
)
check_table

,Requirement,Status
0,Saved model exists,PASS
1,Saved vectorizer exists,PASS
2,preprocessing.py exists,PASS
3,requirements.txt exists,PASS
4,Model loaded,PASS
5,Vectorizer loaded,PASS
6,Known evaluation reproduced,PASS


In [16]:
all_passed = all(checks.values())
print("DAY 1 STATUS:", "READY FOR FASTAPI" if all_passed else "FIX FAILED CHECKS")

DAY 1 STATUS: READY FOR FASTAPI


<a id="section13"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">13. What I Learned Today</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<table style="width:100%;border-collapse:collapse;">
<tr style="background-color:#0F766E;color:white;"><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Concept</th><th style="padding:9px;border:1px solid #CBD5E1;color:white;">Meaning in our project</th></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Deployment</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Move the trained model from a notebook toward a real user-facing application.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Serialization</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Save trained objects so they can be loaded without retraining.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Preprocessing artifact</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Reuse the same text cleaning and TF-IDF transformation from Week 8.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Training/serving skew</b></td><td style="padding:9px;border:1px solid #CBD5E1;">A mismatch between how training data and deployment input are prepared.</td></tr>
<tr><td style="padding:9px;border:1px solid #CBD5E1;"><b>Reproducibility</b></td><td style="padding:9px;border:1px solid #CBD5E1;">Use fixed seeds, pinned dependencies, and traceable experiment information.</td></tr>
</table>
<div style="font-size:1.08em;font-weight:bold;text-align:center;background-color:#FFFBEB;border:1px solid #FBBF24;padding:14px;border-radius:8px;margin:16px 0;color:#111827;">Final Day 1 flow: Week 8 trained model → save/load artifacts → same preprocessing → verified prediction → reproducible deployment package.</div>
</div>

<a id="section14"></a>
<h2 style="color:#FFFFFF;background:linear-gradient(90deg,#0F766E,#0284C7);font-weight:bold;margin-top:30px;padding:10px 15px;border-radius:8px;">14. Day 2 Preview — FastAPI</h2>

<div style="background-color:#F8F8F8;color:#111111;border:1px solid #E9E9E9;border-radius:9px;padding:16px 18px;margin:10px 0 18px 0;line-height:1.7;">
<p>Day 1 prepared the artifacts. Day 2 will place the same prediction logic behind a REST API.</p>
<div style="background-color:#CCFBF1;color:#111827;padding:12px 14px;border-radius:8px;border:1px solid #CBD5E1;font-family:Consolas,'Courier New',monospace;font-size:0.98em;font-weight:600;line-height:1.6;white-space:pre-wrap;margin:10px 0;">User / Client
   ↓ POST /predict
FastAPI
   ↓
preprocess_to_string()
   ↓
TF-IDF vectorizer
   ↓
Logistic Regression
   ↓
JSON response</div>
<div style="border-left:5px solid #FBBF24;background-color:#FFFBEB;padding:11px 15px;margin:12px 0;border-radius:5px;color:#78350F;"><b>Next question:</b> How can another website, mobile app, or backend send a review to our model and receive the prediction as JSON?</div>
</div>

<div style="text-align:center; margin-top:28px; color:#475569;">
<b>End of Week 9 — Day 1: Sprint 4 Planning, Serialization & MLOps</b><br>
Week 8 trained model → Deployment-ready artifacts → Ready for FastAPI
</div>